# Hotel Booking Data Analysis
**Group members:** Allan Jakubovits, Name 2, Name 3, Name 4

## 1. Context and Research Questions

**What one row represents:** one hotel booking (completed or cancelled).

**Population and period:** 1,000 sampled bookings from a city hotel (Lisbon) and a resort hotel (Algarve), Portugal, with arrival dates from July 2015 to August 2017. Source: António, de Almeida & Nunes (2019), *Hotel booking demand datasets*, Data in Brief, 22, 41–49.

**Purpose:** to describe booking patterns, cancellations, and prices at the two hotels and to give evidence-based suggestions to hotel management. The analysis describes associations only, not causes.


**Research questions**
1. Do cancellation rates differ across market segments?
2. Is the number of guests in a booking associated with the average daily rate?
3. How does the average daily rate vary by arrival month in each hotel?

## 2. Imports and display settings

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [9]:

# 1. Load the dataset
df = pd.read_csv("hotel_bookings_final_project.csv")

# 2. Basic shape
print("Shape (rows, columns):", df.shape)

# 3. Column names and data types
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)





Shape (rows, columns): (1000, 20)

Column names:
['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'country', 'market_segment', 'deposit_type', 'customer_type', 'average_daily_rate', 'is_repeated_guest', 'booking_changes', 'required_car_parking_spaces', 'total_of_special_requests']

Data types:
hotel                              str
is_canceled                      int64
lead_time                        int64
arrival_date_year                int64
arrival_date_month                 str
arrival_date_day_of_month        int64
stays_in_weekend_nights          int64
stays_in_week_nights             int64
adults                           int64
children                         int64
babies                           int64
country                            str
market_segment                     str
deposit_type                       str
customer_type  

In [12]:
# 4. Preview of the dataset
print("\nHead (first 5 rows):")
print(df.head())






Head (first 5 rows):
          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  arrival_date_day_of_month  stays_in_weekend_nights  \
0    City Hotel            1          1               2015          September                         30                        0   
1  Resort Hotel            1         19               2016              March                         19                        2   
2  Resort Hotel            0          9               2017             August                          1                        0   
3  Resort Hotel            0        110               2016           November                         11                        0   
4    City Hotel            0        329               2017               July                         27                        0   

   stays_in_week_nights  adults  children  babies country market_segment deposit_type customer_type  average_daily_rate  \
0                     2       1         0       0     PR

In [13]:
print("\nRandom sample:")
print(df.sample(5, random_state=42))

# 5. Missing values
print("\nMissing values per column:")
print(df.isna().sum())




Random sample:
            hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  arrival_date_day_of_month  stays_in_weekend_nights  \
521    City Hotel            0        190               2016           December                         28                        0   
737  Resort Hotel            0        225               2016                May                         26                        0   
740    City Hotel            1        105               2017               June                         26                        1   
660    City Hotel            0        198               2017              March                         17                        1   
411  Resort Hotel            0         54               2016            January                          9                        2   

     stays_in_week_nights  adults  children  babies country market_segment deposit_type    customer_type  average_daily_rate  \
521                     4       2         0  

In [15]:
# 6. Exact duplicates
print("\nNumber of exact duplicate rows:", df.duplicated().sum())

# 7. Category counts (for object / categorical columns)
print("\nCategory counts:")
for col in df.select_dtypes(include=["object", "string", "category"]).columns:
    print(f"\nColumn: {col}")
    print(df[col].value_counts())





Number of exact duplicate rows: 26

Category counts:

Column: hotel
hotel
City Hotel      642
Resort Hotel    358
Name: count, dtype: int64

Column: arrival_date_month
arrival_date_month
May          126
August       115
July         111
October       91
June          90
April         88
September     76
December      75
February      68
March         65
November      50
January       45
Name: count, dtype: int64

Column: country
country
PRT    372
GBR    125
ESP     78
FRA     76
DEU     66
IRL     36
ITA     31
USA     24
BEL     21
BRA     20
NLD     18
CHE     18
AUT     11
CN      11
SWE     10
TUR      8
POL      6
ISR      6
NOR      5
AUS      5
FIN      5
CHN      5
DNK      3
LUX      3
ROU      2
RUS      2
KOR      2
IND      2
HUN      2
MAR      2
NZL      2
LVA      1
ISL      1
MEX      1
UKR      1
BGR      1
CAF      1
LBY      1
HRV      1
TWN      1
SGP      1
PHL      1
AGO      1
GIB      1
MOZ      1
GRC      1
ARG      1
EGY      1
PER      1
ARM      1
KIR    

In [16]:
# 8. Numerical summary statistics
print("\nNumerical summary statistics:")
print(df.describe())


Numerical summary statistics:
       is_canceled  lead_time  arrival_date_year  arrival_date_day_of_month  stays_in_weekend_nights  stays_in_week_nights  adults  \
count      1000.00    1000.00            1000.00                    1000.00                  1000.00               1000.00 1000.00   
mean          0.37     105.06            2016.13                      16.13                     0.92                  2.52    1.86   
std           0.48     106.45               0.70                       8.96                     0.95                  1.80    0.53   
min           0.00       0.00            2015.00                       1.00                     0.00                  0.00    0.00   
25%           0.00      19.00            2016.00                       8.00                     0.00                  1.00    2.00   
50%           0.00      69.50            2016.00                      16.00                     1.00                  2.00    2.00   
75%           1.00     157.00  

## 3. Assess and clean the data

# Cleaning Log

| Issue checked | Evidence | Decision | Reason |
|--------------|----------|----------|--------|
| Missing values | `country` has 2 missing values | Fill with `"Unknown"` | Very small number of missing values; dropping rows would lose data unnecessarily |
| Exact duplicates | 26 duplicate rows found (`df.duplicated().sum()`) | Remove duplicates | Duplicates distort statistics and model training |
| Invalid date combinations | All `arrival_date_month` values match valid month names | Keep | No invalid months detected |
| Impossible negative values | Checked numeric columns; no values < 0 | Keep | Dataset contains no impossible negative values |
| Zero total guests | Some rows have adults+children+babies = 0 | Keep for now | Could be test bookings or errors; unclear without domain knowledge |
| Zero total nights | Some rows have weekend+nights = 0 | Keep | Same‑day stays are possible; not necessarily invalid |
| Unusual lead_time | Values up to 542 days | Keep | Long lead times are realistic for resort bookings and group travel |
| Unusual ADR | Maximum ADR = 304 | Keep | High ADR values occur in luxury or peak‑season bookings |


In [18]:

# Load original dataset
df = pd.read_csv("hotel_bookings_final_project.csv")

# Create cleaned copy
cleaned_df = df.copy()

# --- 1. Missing values ---
cleaned_df['country'] = cleaned_df['country'].fillna("Unknown")

# --- 2. Exact duplicates ---
cleaned_df = cleaned_df.drop_duplicates()

# --- 3. Invalid date combinations ---
valid_months = [
    "January","February","March","April","May","June",
    "July","August","September","October","November","December"
]
invalid_months = cleaned_df[~cleaned_df['arrival_date_month'].isin(valid_months)]

# --- 4. Impossible negative values ---
negative_values = cleaned_df[
    (cleaned_df[['lead_time','stays_in_weekend_nights','stays_in_week_nights',
                 'adults','children','babies','average_daily_rate']] < 0).any(axis=1)
]

# --- 5. Bookings with zero total guests ---
cleaned_df['total_guests'] = cleaned_df['adults'] + cleaned_df['children'] + cleaned_df['babies']
zero_guest_rows = cleaned_df[cleaned_df['total_guests'] == 0]

# --- 6. Bookings with zero total nights ---
cleaned_df['total_nights'] = cleaned_df['stays_in_weekend_nights'] + cleaned_df['stays_in_week_nights']
zero_night_rows = cleaned_df[cleaned_df['total_nights'] == 0]

# --- 7. Unusual lead_time values ---
unusual_lead_time = cleaned_df[cleaned_df['lead_time'] > 365]

# --- 8. Unusual ADR values ---
unusual_adr = cleaned_df[cleaned_df['average_daily_rate'] > 500]


In [22]:
cleaned_df.shape

(974, 22)

## 4. Transform and enrich the data

In [ ]:

# 1. Proper arrival_date
cleaned_df['arrival_date'] = pd.to_datetime(
    cleaned_df['arrival_date_year'].astype(str) + '-' +
    cleaned_df['arrival_date_month'].astype(str) + '-' +
    cleaned_df['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)

# 2. total_nights (already created earlier, but ensure it's correct)
cleaned_df['total_nights'] = (
    cleaned_df['stays_in_weekend_nights'] +
    cleaned_df['stays_in_week_nights']
)

# 3. total_guests (already created earlier)
cleaned_df['total_guests'] = (
    cleaned_df['adults'] +
    cleaned_df['children'] +
    cleaned_df['babies']
)

# 4. estimated_booking_value
cleaned_df['estimated_booking_value'] = (
    cleaned_df['average_daily_rate'] * cleaned_df['total_nights']
)
